# 01 — Générer les CSV métriques à partir des traces Nowledgeable

Ce notebook détecte et exécute **tous les scripts Python** présents dans
`scripts_Nowledgeable/`. Les métriques produites sont enregistrées dans le
dossier `csv/`, avec un journal d'exécution par script dans `csv/logs/`.

Par défaut, le suffixe `_Nowledgeable` est ajouté aux noms de fichiers afin
d'éviter d'écraser les métriques de même nom produites par Mirabelle ou
ProgSnap2.

## 1. Configuration

Les paramètres ci-dessous peuvent être modifiés avant l'exécution.

In [ ]:
from pathlib import Path
import sys
import os
import runpy
import contextlib
import traceback
import pandas as pd
from IPython.display import display


def detect_project_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "Chaine",
        Path.cwd() / "chaine_extracted" / "Chaine",
        Path.cwd().parent / "Chaine",
    ]
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "scripts_Nowledgeable").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Impossible de détecter automatiquement le dossier Chaine. "
        "Modifiez PROJECT_DIR manuellement dans cette cellule."
    )


# PROJECT_DIR = Path(r"C:/Users/.../Chaine")
PROJECT_DIR = detect_project_dir()

DATA_DIR = PROJECT_DIR / "data"
SCRIPTS_DIR = PROJECT_DIR / "scripts_Nowledgeable"
CSV_DIR = PROJECT_DIR / "csv"
LOG_DIR = CSV_DIR / "logs"

# Fichier de traces Nowledgeable à traiter.
INPUT_FILE = DATA_DIR / "session_13568_answers_corrige.csv"

# Suffixe ajouté au nom des CSV pour éviter les collisions avec les autres jeux
# de données. Utilisez OUTPUT_SUFFIX = "" pour conserver les noms par défaut.
OUTPUT_SUFFIX = "_Nowledgeable"

# Si True, supprime les CSV ciblés avant de les régénérer.
OVERWRITE_OUTPUTS = True

CSV_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Projet :", PROJECT_DIR)
print("Entrée :", INPUT_FILE)
print("Scripts :", SCRIPTS_DIR)
print("Sorties CSV :", CSV_DIR)

## 2. Vérification du fichier d'entrée

Le notebook vérifie la présence du fichier et affiche un aperçu de ses
colonnes. Chaque script est ensuite exécuté seulement si ses colonnes requises
sont disponibles.

In [ ]:
if not SCRIPTS_DIR.exists():
    raise FileNotFoundError(f"Dossier scripts_Nowledgeable introuvable : {SCRIPTS_DIR}")

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Fichier de traces introuvable : {INPUT_FILE}")

input_df = pd.read_csv(
    INPUT_FILE,
    dtype={"studentId": "string"},
    on_bad_lines="skip",
    engine="python",
)

print(f"Fichier chargé : {input_df.shape[0]:,} lignes × {input_df.shape[1]:,} colonnes")
display(pd.DataFrame({
    "colonne": input_df.columns,
    "non_null": [input_df[c].notna().sum() for c in input_df.columns],
    "dtype": [str(input_df[c].dtype) for c in input_df.columns],
}))

## 3. Détection des scripts à exécuter

Tous les fichiers `*.py` du dossier sont détectés automatiquement. La table de
métadonnées ci-dessous fournit les noms de sortie et les colonnes requises pour
les scripts actuellement présents. Un futur script absent de cette table sera
tout de même exécuté, avec un nom de sortie dérivé de son nom de fichier.

In [ ]:
SCRIPT_METADATA = {
    "eq_Nowledgeable.py": {
        "output": "ErrorQuotient.csv",
        "description": "Error Quotient calculé sur les erreurs de compilation",
        "required": ["studentId", "exerciceId", "answeredAt", "recordedFeedback"],
    },
    "max_unchanged_code_attempts_Nowledgeable.py": {
        "output": "MaxUnchangedCodeAttempts.csv",
        "description": "Plus longue série de soumissions consécutives sans changement de code",
        "required": ["studentId", "exerciceId", "answeredAt", "answerContent"],
    },
    "red_Nowledgeable.py": {
        "output": "RED.csv",
        "description": "Repeated Error Density",
        "required": ["studentId", "exerciceId", "answeredAt", "recordedFeedback"],
    },
    "score_progression_Nowledgeable.py": {
        "output": "ScoreProgression.csv",
        "description": "Progression moyenne du score entre première et dernière tentative",
        "required": ["studentId", "exerciceId", "answeredAt", "answerScore"],
    },
    "testratio_Nowledgeable.py": {
        "output": "TestPassRate.csv",
        "description": "Ratio de cas de test réussis",
        "required": ["studentId", "recordedFeedback"],
    },
}


def add_suffix(filename: str, suffix: str) -> str:
    path = Path(filename)
    return f"{path.stem}{suffix}{path.suffix}"


script_paths = sorted(
    path for path in SCRIPTS_DIR.glob("*.py")
    if not path.name.startswith("_")
)

SCRIPT_JOBS = []
for script_path in script_paths:
    metadata = SCRIPT_METADATA.get(script_path.name, {})
    default_output = metadata.get("output", f"{script_path.stem}.csv")
    SCRIPT_JOBS.append({
        "script": script_path.name,
        "output": add_suffix(default_output, OUTPUT_SUFFIX),
        "description": metadata.get(
            "description",
            "Script détecté automatiquement (métadonnées non renseignées)",
        ),
        "required": metadata.get("required", []),
    })

if not SCRIPT_JOBS:
    raise FileNotFoundError(f"Aucun script Python trouvé dans : {SCRIPTS_DIR}")

jobs_df = pd.DataFrame([
    {k: v for k, v in job.items() if k != "required"}
    for job in SCRIPT_JOBS
])
display(jobs_df)

## 4. Exécution des scripts

Chaque script reçoit deux arguments : le chemin du CSV d'entrée et le chemin
du CSV de sortie. La sortie standard et les erreurs sont enregistrées dans un
fichier de log dédié.

In [ ]:
def missing_columns(job, columns):
    return [column for column in job.get("required", []) if column not in columns]


def output_summary(path: Path):
    if not path.exists():
        return {"rows": None, "columns": None}
    try:
        df = pd.read_csv(path)
        return {
            "rows": int(df.shape[0]),
            "columns": ", ".join(df.columns.astype(str).tolist()),
        }
    except Exception as exc:
        return {"rows": None, "columns": f"lecture impossible : {exc}"}


@contextlib.contextmanager
def temporary_cwd(path: Path):
    old_cwd = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(old_cwd)


if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

report = []
available_columns = set(input_df.columns)

for job in SCRIPT_JOBS:
    print(f"Lancement : {job['script']} → {job['output']}", flush=True)

    script_path = SCRIPTS_DIR / job["script"]
    output_path = CSV_DIR / job["output"]
    log_path = LOG_DIR / f"{Path(job['script']).stem}.log"

    row = {
        "script": job["script"],
        "output": job["output"],
        "description": job["description"],
        "status": None,
        "returncode": None,
        "output_path": str(output_path),
        "rows": None,
        "columns": None,
        "log_path": str(log_path),
        "message": "",
    }

    missing = missing_columns(job, available_columns)
    if missing:
        row.update(
            status="skipped_missing_columns",
            returncode=None,
            message=f"Colonnes manquantes : {missing}",
        )
        report.append(row)
        print(f"Terminé : {job['script']} — {row['status']} ({row['message']})", flush=True)
        continue

    if OVERWRITE_OUTPUTS and output_path.exists():
        output_path.unlink()

    argv = [str(script_path), str(INPUT_FILE), str(output_path)]
    old_argv = sys.argv[:]

    try:
        with log_path.open("w", encoding="utf-8") as log_file:
            log_file.write("COMMANDE LOGIQUE\npython " + " ".join(argv) + "\n\nSORTIE DU SCRIPT\n")
            log_file.flush()
            with temporary_cwd(PROJECT_DIR), contextlib.redirect_stdout(log_file), contextlib.redirect_stderr(log_file):
                sys.argv = argv
                try:
                    runpy.run_path(str(script_path), run_name="__main__")
                    returncode = 0
                except SystemExit as exc:
                    returncode = int(exc.code or 0) if isinstance(exc.code, int) else 1

        summary = output_summary(output_path)
        status = "ok" if returncode == 0 and output_path.exists() else "error"
        message = "CSV produit" if output_path.exists() else "Aucun CSV produit"
    except Exception as exc:
        returncode = 1
        status = "error"
        message = f"{type(exc).__name__}: {exc}"
        with log_path.open("a", encoding="utf-8") as log_file:
            log_file.write("\n\nEXCEPTION NOTEBOOK\n")
            traceback.print_exc(file=log_file)
        summary = output_summary(output_path)
    finally:
        sys.argv = old_argv

    row.update(
        status=status,
        returncode=returncode,
        rows=summary["rows"],
        columns=summary["columns"],
        message=message,
    )
    report.append(row)
    print(f"Terminé : {job['script']} — {row['status']} ({row['message']})", flush=True)

report_df = pd.DataFrame(report)
report_path = LOG_DIR / "run_report_Nowledgeable.csv"
report_df.to_csv(report_path, index=False)

display(report_df)
print(f"Rapport sauvegardé : {report_path}")

## 5. Contrôle des sorties

In [ ]:
generated = []
for job in SCRIPT_JOBS:
    csv_path = CSV_DIR / job["output"]
    if not csv_path.exists():
        generated.append({
            "fichier": csv_path.name,
            "statut": "absent",
            "lignes": None,
            "colonnes": None,
            "noms_colonnes": "",
        })
        continue

    try:
        df = pd.read_csv(csv_path)
        generated.append({
            "fichier": csv_path.name,
            "statut": "ok",
            "lignes": df.shape[0],
            "colonnes": df.shape[1],
            "noms_colonnes": ", ".join(df.columns.astype(str).tolist()),
        })
    except Exception as exc:
        generated.append({
            "fichier": csv_path.name,
            "statut": "lecture impossible",
            "lignes": None,
            "colonnes": None,
            "noms_colonnes": str(exc),
        })

generated_df = pd.DataFrame(generated)
display(generated_df)

## 6. Affichage des journaux des erreurs éventuelles

In [ ]:
error_rows = report_df[report_df["status"] != "ok"].copy()

if error_rows.empty:
    print("Aucune erreur détectée.")
else:
    display(error_rows[["script", "status", "message", "log_path"]])
    for _, row in error_rows.iterrows():
        log_path = Path(row["log_path"])
        print("\n" + "=" * 100)
        print(f"{row['script']} — {row['status']}")
        print(row["message"])
        if log_path.exists():
            log_text = log_path.read_text(encoding="utf-8", errors="replace")
            print(log_text[-4000:])